<a href="https://colab.research.google.com/github/heyanugrah/Transformers/blob/main/onlydecoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch
import torch
from torch import nn
import math
import torch.nn.functional as F

In [ ]:
# class Decoder(nn.Module) :
#     def __init__(self,d_head,seq_len):
#       super().__init__()
#       self.d_head = d_head
#       self.seq_len = seq_len

#       #masked-multihead parameters
#       self.Wk = nn.Linear(d_head,d_head)
#       self.Wq = nn.Linear(d_head,d_head)
#       self.Wv = nn.Linear(d_head,d_head)
#       self.norm1 = nn.LayerNorm(d_head)

#       self.ffn = nn.Sequential(
#           nn.Linear(d_head,d_head*4),
#           nn.GELU(),
#           nn.Linear(d_head*4,d_head)
#       )

#       #multihead parameters
#       self.Wk_multi = nn.Linear(d_head,d_head)
#       self.Wq_multi = nn.Linear(d_head,d_head)
#       self.Wv_multi = nn.Linear(d_head,d_head)
#       self.norm2 = nn.LayerNorm(d_head)

#       #final
#       self.norm3 = nn.LayerNorm(d_head)
#       self.LinearLayer = nn.Linear(d_head,d_head)

#     def maskedSelfAttention(self,input_embedings_ids,padding_mask=None):

#       K = self.Wk(input_embedings_ids)
#       Q = self.Wq(input_embedings_ids)
#       V = self.Wv(input_embedings_ids)

#       attention_scores = Q @ K.transpose(-2,-1) #row become column and column become row

#       scaled_score = attention_scores / math.sqrt(K.shape[-1])

#       if padding_mask is not None:
#         scaled_score = scaled_score.masked_fill(
#              padding_mask == 0,
#             -1e9
#           )

#       look_ahead_matrix = torch.triu(torch.ones(self.seq_len,self.seq_len),dim=1)

#       look_ahead_mask = look_ahead_matrix.masked_fill(
#           look_ahead_matrix == 1,
#           -1e9
#       )

#       combined_mask = scaled_score + look_ahead_mask
#       attention_weight = F.softmax(combined_mask,dim =-1)
#       final_attention = attention_weight @ V

#       return final_attention


#     def crossMultiheadSelfAttention(self,encoder_output,normalized,padding_mask=None):

#       k_multi = self.Wk_multi(encoder_output)
#       q_multi = self.Wq_multi(normalized)
#       v_multi = self.Wv_multi(encoder_output)

#       attention_scores = q_multi @ k_multi.transpose(-2,-1)
#       scaled_score = attention_scores / math.sqrt(k_multi.shape[-1])

#       if padding_mask is not None:
#         scaled_score = scaled_score.masked_fill(
#             padding_mask == 0,
#             -1e9
#         )

#       attention_weight = F.softmax(scaled_score,dim =-1)
#       final_attention = attention_weight @ v_multi

#       return final_attention


#     def forward(self,input_embedings_ids,encoder_output,padding_mask=None):

#       #masked-multi-head-attention
#       attention_output = self.maskedSelfAttention(
#           input_embedings_ids,
#           padding_mask
#         )

#       #Add Residual connection
#       residual_1 = attention_output + input_embedings_ids

#       #normalize
#       masked_multihead_output = self.norm1(residual_1)

#       #multi-head attention
#       cross_multihead_attention = self.crossMultiheadSelfAttention(
#           encoder_output,
#           masked_multihead_output,
#           padding_mask
#         )

#       #Add Residual connection
#       residual_2 = cross_multihead_attention + masked_multihead_output

#       #normalize
#       normalized2 = self.norm2(residual_2)

#       #feed-forward-network
#       ffn = self.ffn(normalized2)

#       #Add Residual connection
#       residual_3 = ffn + normalized2

#       #final output
#       final_normalized_output = self.norm3(residual_3)

#       linear_output = self.LinearLayer(final_normalized_output)
#       return linear_output


In [8]:
import torch
from torch import nn
import torch.nn.functional as F
import math

class Decoder(nn.Module):

    def __init__(self, d_model, seq_len, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.seq_len = seq_len
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # -----------------------------
        # Q, K, V Projection
        # -----------------------------
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        # Output Projection
        self.out = nn.Linear(d_model, d_model)

        # Layer Normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)

        # --------------------------------
        # Register causal mask once
        # --------------------------------
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool()

        self.register_buffer("causal_mask", causal_mask)

    def maskedSelfAttention(self, x, padding_mask=None):

        batch_size, seq_len, _ = x.size()

        # -----------------------------
        # Q K V
        # -----------------------------
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # -----------------------------
        # Split Heads
        # -----------------------------
        Q = Q.view(batch_size, seq_len,
                   self.num_heads,
                   self.head_dim).transpose(1, 2)

        K = K.view(batch_size, seq_len,
                   self.num_heads,
                   self.head_dim).transpose(1, 2)

        V = V.view(batch_size, seq_len,
                   self.num_heads,
                   self.head_dim).transpose(1, 2)

        # -----------------------------
        # Attention Scores
        # -----------------------------
        scores = Q @ K.transpose(-2, -1)

        scores = scores / math.sqrt(self.head_dim)

        # -----------------------------
        # Padding Mask
        # -----------------------------
        if padding_mask is not None:

            if padding_mask.dim() == 2:
                padding_mask = padding_mask.unsqueeze(1).unsqueeze(2)

            scores = scores.masked_fill(
                padding_mask == 0,
                -1e9
            )

        # -----------------------------
        # Causal Mask
        # -----------------------------
        causal_mask = self.causal_mask[:seq_len, :seq_len]

        scores = scores.masked_fill(
            causal_mask,
            -1e9
        )

        # -----------------------------
        # Attention Weights
        # -----------------------------
        attention = F.softmax(scores, dim=-1)

        attention = self.dropout(attention)

        # -----------------------------
        # Weighted Sum
        # -----------------------------
        output = attention @ V

        # -----------------------------
        # Merge Heads
        # -----------------------------
        output = output.transpose(1, 2).contiguous()

        output = output.view(
            batch_size,
            seq_len,
            self.d_model
        )

        output = self.out(output)

        return output

    def forward(self, x, padding_mask=None):

        # -----------------------------
        # Pre-LayerNorm Self Attention
        # -----------------------------
        x = x + self.maskedSelfAttention(
            self.norm1(x),
            padding_mask
        )

        # -----------------------------
        # Pre-LayerNorm Feed Forward
        # -----------------------------
        x = x + self.ffn(
            self.norm2(x)
        )

        return x

In [9]:
text = """
Artificial intelligence is changing the world.
Deep learning models can understand language.
Transformers are powerful neural networks.
"""

In [10]:
from collections import Counter

words = text.lower().split()

counter = Counter(words)

vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for word in counter:
    vocab[word] = len(vocab)

inverse_vocab = {v:k for k,v in vocab.items()}

In [11]:
tokens = [vocab[word] for word in words]

print(tokens)

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


In [12]:
import torch
from torch.utils.data import Dataset

class GPTDataset(Dataset):

    def __init__(self, tokens, seq_len):

        self.samples = []

        for i in range(len(tokens)-seq_len):

            x = tokens[i:i+seq_len]

            y = tokens[i+1:i+seq_len+1]

            self.samples.append((x,y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,index):

        x,y=self.samples[index]

        return (
            torch.tensor(x),
            torch.tensor(y)
        )

In [13]:
seq_len = 8
dataset = GPTDataset(tokens,seq_len)

In [14]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [15]:
import torch
from torch import nn

class GPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        seq_len,
        heads,
        layers
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position = nn.Embedding(
            seq_len,
            d_model
        )

        self.decoder = nn.ModuleList([
            Decoder(
                d_model,
                seq_len,
                heads
            )
            for _ in range(layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self,x):

        B,S = x.shape

        pos = torch.arange(
            S,
            device=x.device
        ).unsqueeze(0)

        x = self.embedding(x)

        x = x + self.position(pos)

        for layer in self.decoder:
            x = layer(x)

        x = self.norm(x)

        logits = self.lm_head(x)

        return logits

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT(
    vocab_size=len(vocab),
    d_model=128,
    seq_len=seq_len,
    heads=4,
    layers=4
)

model.to(device)

GPT(
  (embedding): Embedding(19, 128)
  (position): Embedding(8, 128)
  (decoder): ModuleList(
    (0-3): 4 x Decoder(
      (Wq): Linear(in_features=128, out_features=128, bias=True)
      (Wk): Linear(in_features=128, out_features=128, bias=True)
      (Wv): Linear(in_features=128, out_features=128, bias=True)
      (out): Linear(in_features=128, out_features=128, bias=True)
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=512, out_features=128, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=128, out_features=19, bias=True)
)

In [17]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [26]:
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0
    total_correct_predictions = 0
    total_samples = 0

    for x,y in loader:

        x = x.to(device)

        y = y.to(device)

        logits = model(x)

        loss = criterion(
            logits.reshape(-1,len(vocab)),
            y.reshape(-1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        # Calculate accuracy
        predictions = torch.argmax(logits, dim=-1)
        total_correct_predictions += (predictions == y).sum().item()
        total_samples += y.numel() # Total number of elements in y

    avg_loss = total_loss / len(loader)
    accuracy = total_correct_predictions / total_samples

    print(
        f"Epoch {epoch+1}/{epochs}: Loss = {avg_loss:.4f}, Accuracy = {accuracy:.4f}"
    )






Epoch 1/20: Loss = 0.0268, Accuracy = 1.0000
Epoch 2/20: Loss = 0.0258, Accuracy = 1.0000
Epoch 3/20: Loss = 0.0260, Accuracy = 1.0000
Epoch 4/20: Loss = 0.0246, Accuracy = 1.0000
Epoch 5/20: Loss = 0.0242, Accuracy = 1.0000
Epoch 6/20: Loss = 0.0237, Accuracy = 1.0000
Epoch 7/20: Loss = 0.0250, Accuracy = 1.0000
Epoch 8/20: Loss = 0.0242, Accuracy = 1.0000
Epoch 9/20: Loss = 0.0227, Accuracy = 1.0000
Epoch 10/20: Loss = 0.0236, Accuracy = 1.0000
Epoch 11/20: Loss = 0.0218, Accuracy = 1.0000
Epoch 12/20: Loss = 0.0224, Accuracy = 1.0000
Epoch 13/20: Loss = 0.0200, Accuracy = 1.0000
Epoch 14/20: Loss = 0.0210, Accuracy = 1.0000
Epoch 15/20: Loss = 0.0193, Accuracy = 1.0000
Epoch 16/20: Loss = 0.0190, Accuracy = 1.0000
Epoch 17/20: Loss = 0.0185, Accuracy = 1.0000
Epoch 18/20: Loss = 0.0185, Accuracy = 1.0000
Epoch 19/20: Loss = 0.0195, Accuracy = 1.0000
Epoch 20/20: Loss = 0.0175, Accuracy = 1.0000

Generated text: artificial intelligence is changing the world. deep
Generated text: deep

In [27]:
def predict_sentence(model, start_text, num_words_to_generate, seq_len, vocab, inverse_vocab, device):
    model.eval()
    with torch.no_grad():
        # Tokenize the start text
        input_tokens = [vocab.get(word.lower(), vocab['<UNK>']) for word in start_text.lower().split()]
        generated_tokens = list(input_tokens)

        for _ in range(num_words_to_generate):
            # Ensure the input sequence for the model is of seq_len
            current_input = generated_tokens[-seq_len:] if len(generated_tokens) > seq_len else generated_tokens

            if not current_input:
                break # Cannot predict if there's no input

            input_tensor = torch.tensor(current_input).unsqueeze(0).to(device)

            # Get model prediction
            logits = model(input_tensor)

            # Get the logits for the last token in the sequence
            next_token_logits = logits[0, -1, :]

            # Predict the next token
            predicted_token_id = torch.argmax(next_token_logits).item()

            # Add the predicted token to the generated sequence
            generated_tokens.append(predicted_token_id)

            # Break if it predicts a padding or unknown token (or end token if we had one)
            if predicted_token_id == vocab['<PAD>'] or predicted_token_id == vocab['<UNK>']:
                break

        # Convert token IDs back to words
        generated_words = [inverse_vocab.get(token_id, '<UNK>') for token_id in generated_tokens]
        return ' '.join(generated_words)

In [25]:
# Example usage after training
start_phrase = "Artificial intelligence"
num_words = 5 # Number of words to generate after the start phrase
predicted_text = predict_sentence(model, start_phrase, num_words, seq_len, vocab, inverse_vocab, device)
print(f"\nGenerated text: {predicted_text}")

start_phrase_2 = "Deep learning"
predicted_text_2 = predict_sentence(model, start_phrase_2, num_words, seq_len, vocab, inverse_vocab, device)
print(f"Generated text: {predicted_text_2}")


Generated text: artificial intelligence is changing the world. deep
Generated text: deep learning models can understand language. transformers
